# ALPN Glomerulus Volume Mapping — MCNS + FAFB

Convex-hull volumes per antennal-lobe glomerulus, **split by AL side (R/L)**, from ALPN postsynaptic sites (PN inputs) and ORN presynaptic sites (ORN outputs).

Run the next cell to define everything, then the last cell to execute. Set `GLOMERULUS_LIMIT` (in the config) to a small int for a quick test.

In [1]:
#!/usr/bin/env python3
"""
ALPN Glomerulus Volume Mapping — MCNS + FAFB
==============================================
Computes 3-D convex-hull volumes for every antennal-lobe glomerulus, split by
antennal-lobe side (AL_R / AL_L), using two complementary synapse sets:

  (A) ALPN postsynaptic sites  ("PN inputs")
        · Input sites on the antennal-lobe projection neurons.
        · Glomerulus identity comes directly from the ALPN cell type.

  (B) ORN presynaptic sites  ("ORN outputs")
        · ORN → ALPN presynapse locations (dense; tile the glomerular neuropil).
        · Each presynapse is assigned to a glomerulus + AL side via its
          postsynaptic ALPN partner.  ORNs project *bilaterally*, so the
          post-partner (ALPNs are unilateral) is what cleanly separates the
          left- and right-AL clouds.

Why group by the post-partner ALPN?
-----------------------------------
ORN somata sit on one side but their axons innervate *both* antennal lobes, so
grouping ORN presynapses by ORN soma-side merges the two ALs into one ~200 µm
blob.  The cognate ALPN is unilateral, so its (glomerulus, side) is the correct
label for the presynapse and naturally splits AL_R from AL_L.

Datasets  (verified live, 2026-06)
----------------------------------
  MCNS  — Male CNS connectome → neuprint-python
          https://neuprint.janelia.org   dataset: male-cns:v0.9
          coordinates in 8 nm voxels.
  FAFB  — FlyWire / Full Adult Female Brain → CAVEclient
          datastack: flywire_fafb_public   materialization: 783
          coordinates already in nm.  Cell typing from the public Codex
          classification dump (data/FlyWire/classification.csv.gz, v783).

Tokens
------
  NEUPRINT_TOKEN  — read from a .env file at the repo root (python-dotenv) or
                    the environment.  https://neuprint.janelia.org/account
  CAVE token      — taken from the locally cached CAVE secret
                    (~/.cloudvolume/secrets).  No env var needed; if you have
                    never authenticated run:  caveclient + client.auth ... .

Install
-------
pip install neuprint-python caveclient python-dotenv numpy pandas scipy matplotlib
"""

from __future__ import annotations
import os
import sys
import json
import time
import warnings
from pathlib import Path

# ── Import-path guard ─────────────────────────────────────────────────────────
# A data folder named ``neuprint/`` lives next to the analysis notebooks.  When
# this code runs from that directory (e.g. inside Jupyter), the notebook's own
# directory sits first on ``sys.path`` and that local folder shadows the
# pip-installed ``neuprint-python`` package.  De-prioritise (don't remove) the
# current directory so installed packages win, while local helper modules
# remain importable as a fallback.
_cwd = os.path.realpath(os.getcwd())
sys.path = ([p for p in sys.path if os.path.realpath(p or ".") != _cwd]
            + [p for p in sys.path if os.path.realpath(p or ".") == _cwd])

import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ── MCNS / neuprint ───────────────────────────────────────────────────────────
NP_SERVER   = "https://neuprint.janelia.org"
NP_DATASET  = "male-cns:v0.9"
NP_AL_ROIS  = ["AL(R)", "AL(L)"]
NP_VOXEL_NM = 8                        # nm per voxel in the male-cns EM volume

# ── FAFB / FlyWire ────────────────────────────────────────────────────────────
FW_DATASTACK = "flywire_fafb_public"
FW_MAT_VER   = 783                     # public materialisation matching the dump
FW_SYN_TABLE = "synapses_nt_v1"
# Cell typing comes from the Codex classification dump (root_ids match v783):
FW_CLASSIFICATION = "data/FlyWire/classification.csv.gz"
FW_AL_MIDLINE_X   = 520_000           # nm; x < midline → left AL, else right AL.
#                                       Splits synapses by location (not by the
#                                       PN's annotated side) so bilateral PNs
#                                       don't merge the two antennal lobes.

# ── Analysis ──────────────────────────────────────────────────────────────────
SYNAPSE_METHOD   = "both"              # "pn_inputs" | "orn_outputs" | "both"
RUN_DATASETS     = ("MCNS", "FAFB")    # subset to ("MCNS",) etc. to run one
MIN_SYNAPSES     = 25                  # skip a (glomerulus, side) with fewer pts
OUTLIER_QTILE    = 0.05                # drop farthest 5 % (radial) before the hull;
#                                        raise to trim neurite tails harder, lower
#                                        to keep more of the cloud.  Convex-hull
#                                        volume is inherently spread-sensitive.
BATCH_SIZE       = 50                  # ids per CAVE query batch (smaller = gentler)
CAVE_RETRIES     = 5                   # retries per CAVE query (transient 500s)
GLOMERULUS_LIMIT = None                # int → only the first N glomeruli (debug)

OUTPUT_DIR = Path("glomerulus_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# LABELLING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def norm_side(raw) -> str:
    """Normalise an AL-side label to 'R' / 'L' (else the raw string)."""
    s = str(raw)
    if "right" in s or "(R)" in s:
        return "R"
    if "left" in s or "(L)" in s:
        return "L"
    return s


def glom_from_orn(t: str) -> str | None:
    """'ORN_DA1' → 'DA1'."""
    if not isinstance(t, str) or not t.startswith("ORN_"):
        return None
    g = t[len("ORN_"):]
    return g or None


def glom_from_alpn(t: str) -> str | None:
    """'DA1_lPN' → 'DA1'  (uniglomerular-PN prefix)."""
    if not isinstance(t, str) or "_" not in t:
        return None
    g = t.split("_")[0]
    return g or None


# Multiglomerular PNs (hemibrain 'M_*'/'MZ_*' tracts, or comma/'+'-joined
# names like 'VP1d,VP4') innervate many glomeruli; lumping their synapses into
# one pseudo-glomerulus balloons the hull, so they are excluded.
_MULTIGLOM = {"M", "MZ"}


def is_uniglom(g) -> bool:
    if not isinstance(g, str) or g in _MULTIGLOM:
        return False
    if any(ch in g for ch in ",+/ "):
        return False
    if g.startswith("CB") and g[2:].isdigit():     # provisional cell-body ID
        return False
    return True


def _locate(rel: str) -> Path:
    """Find a repo-relative data file whether run from repo root or analysis/."""
    here = Path(rel)
    # ``__file__`` is undefined inside a Jupyter kernel; fall back to cwd.
    _here = Path(globals().get("__file__", ".")).resolve().parent
    for base in (Path.cwd(), Path.cwd().parent, _here):
        cand = base / rel
        if cand.exists():
            return cand
    return here   # let the caller raise a clear FileNotFoundError


# ══════════════════════════════════════════════════════════════════════════════
# GEOMETRY
# ══════════════════════════════════════════════════════════════════════════════

def trimmed_hull(pts: np.ndarray, q: float = OUTLIER_QTILE):
    """
    Drop the farthest fraction q of points (by Euclidean distance from the
    median centre) before fitting the hull.

    A convex hull is extremely sensitive to outliers: a handful of stray
    synapses (along a PN's neurite, or mis-localised) make the hull stretch
    into a long spike.  A radial trim removes those — unlike a per-axis
    quantile trim, it also catches diagonal outliers.
    """
    centre = np.median(pts, axis=0)
    dist   = np.linalg.norm(pts - centre, axis=1)
    thresh = np.quantile(dist, 1.0 - q)
    mask   = dist <= thresh
    p      = pts[mask] if mask.sum() >= 4 else pts
    return ConvexHull(p), p


def compute_glom_volumes(syn_df: pd.DataFrame, coord_unit_nm: float = 1.0) -> dict:
    """
    Trimmed convex-hull volume for every (glomerulus, side) group.

    syn_df must have columns [glomerulus, side, x, y, z] in one coordinate unit.
    Returns dict keyed by (glomerulus, side) → metrics.
    """
    to_um   = coord_unit_nm / 1_000.0
    results: dict = {}

    for (glom, side), grp in syn_df.groupby(["glomerulus", "side"]):
        pts = grp[["x", "y", "z"]].values.astype(np.float64)
        if len(pts) < MIN_SYNAPSES:
            continue
        try:
            hull, pts_t = trimmed_hull(pts)
        except Exception as exc:
            print(f"    ConvexHull failed {glom}/{side}: {exc}")
            continue
        centroid = pts_t.mean(axis=0)
        results[(glom, side)] = dict(
            hull        = hull,
            pts         = pts_t,
            raw_pts     = pts,
            centroid    = centroid,
            centroid_nm = centroid * coord_unit_nm,
            vol_um3     = hull.volume * to_um ** 3,
            area_um2    = hull.area   * to_um ** 2,
            n_syn       = len(pts),
            n_hull      = len(pts_t),
        )
    return results


# ══════════════════════════════════════════════════════════════════════════════
# BACKEND A — MCNS via neuprint-python
# ══════════════════════════════════════════════════════════════════════════════

def run_neuprint(method: str = "both") -> dict[str, pd.DataFrame]:
    from dotenv import load_dotenv, find_dotenv
    from neuprint import (Client, fetch_neurons, fetch_synapses,
                          NeuronCriteria as NC, SynapseCriteria as SC)

    load_dotenv(find_dotenv(usecwd=True) or _locate(".env"))
    token = os.getenv("NEUPRINT_TOKEN")
    if not token:
        raise RuntimeError("NEUPRINT_TOKEN not found in environment or .env")

    print(f"\n{'─'*64}\n  MCNS  ·  {NP_SERVER}  ·  {NP_DATASET}\n{'─'*64}")
    c = Client(NP_SERVER, dataset=NP_DATASET, token=token)
    print("  Connected  ✓")

    # ── neuron → glomerulus maps ──────────────────────────────────────────────
    alpn, _ = fetch_neurons(NC(class_="ALPN"), client=c)
    alpn["glomerulus"] = alpn["type"].map(glom_from_alpn)
    alpn = alpn.dropna(subset=["glomerulus"])
    alpn = alpn[alpn["glomerulus"].map(is_uniglom)]      # uniglomerular only
    alpn_glom = dict(zip(alpn["bodyId"], alpn["glomerulus"]))
    alpn_ids  = alpn["bodyId"].tolist()

    orn, _ = fetch_neurons(NC(type="ORN_.*", regex=True), client=c)
    orn["glomerulus"] = orn["type"].map(glom_from_orn)
    orn = orn.dropna(subset=["glomerulus"])
    orn_glom = dict(zip(orn["bodyId"], orn["glomerulus"]))
    orn_ids  = orn["bodyId"].tolist()
    print(f"  ALPNs: {len(alpn_ids)} ({alpn['glomerulus'].nunique()} gloms)  |  "
          f"ORNs: {len(orn_ids)} ({orn['glomerulus'].nunique()} gloms)")

    if GLOMERULUS_LIMIT:
        keep = sorted(set(alpn["glomerulus"]))[:GLOMERULUS_LIMIT]
        alpn_ids = alpn[alpn["glomerulus"].isin(keep)]["bodyId"].tolist()
        orn_ids  = orn[orn["glomerulus"].isin(keep)]["bodyId"].tolist()
        print(f"  [debug] limited to glomeruli: {keep}")

    frames: dict[str, pd.DataFrame] = {}

    # ── A1: ALPN postsynaptic (input) sites ───────────────────────────────────
    if method in ("pn_inputs", "both"):
        print("  [A1] ALPN postsynaptic sites …")
        syn = fetch_synapses(NC(bodyId=alpn_ids),
                             SC(type="post", rois=NP_AL_ROIS, primary_only=True),
                             client=c)
        df = pd.DataFrame({
            "glomerulus": syn["bodyId"].map(alpn_glom).values,
            "side"      : syn["roi"].map(norm_side).values,
            "x": syn["x"].values, "y": syn["y"].values, "z": syn["z"].values,
        }).dropna(subset=["glomerulus"])
        df = df[df["side"].isin(["R", "L"])]
        df.to_csv(OUTPUT_DIR / "mcns_pn_inputs.csv", index=False)
        frames["pn_inputs"] = df
        print(f"    {len(df):,} synapses")

    # ── A2: ORN presynaptic (output) sites in the AL ──────────────────────────
    #  The synapse ROI (AL(R)/AL(L)) gives the side directly, so bilateral ORN
    #  axons split correctly between the two antennal lobes by location.
    if method in ("orn_outputs", "both"):
        print("  [A2] ORN presynaptic sites in AL …")
        syn = fetch_synapses(NC(bodyId=orn_ids),
                             SC(type="pre", rois=NP_AL_ROIS, primary_only=True),
                             client=c)
        df = pd.DataFrame({
            "glomerulus": syn["bodyId"].map(orn_glom).values,
            "side"      : syn["roi"].map(norm_side).values,
            "x": syn["x"].values, "y": syn["y"].values, "z": syn["z"].values,
        }).dropna(subset=["glomerulus"])
        df = df[df["side"].isin(["R", "L"])]
        df.to_csv(OUTPUT_DIR / "mcns_orn_outputs.csv", index=False)
        frames["orn_outputs"] = df
        print(f"    {len(df):,} ORN presynapses")

    return frames


# ══════════════════════════════════════════════════════════════════════════════
# BACKEND B — FAFB / FlyWire via CAVEclient  (coords in nm)
# ══════════════════════════════════════════════════════════════════════════════

def run_flywire(method: str = "both") -> dict[str, pd.DataFrame]:
    from caveclient import CAVEclient

    print(f"\n{'─'*64}\n  FAFB  ·  {FW_DATASTACK}  ·  mat {FW_MAT_VER}\n{'─'*64}")
    cave = CAVEclient(FW_DATASTACK)          # cached CAVE secret → auth
    print("  Connected  ✓")

    # ── neuron typing from the Codex classification dump (v783 root_ids) ──────
    cls_path = _locate(FW_CLASSIFICATION)
    if not cls_path.exists():
        raise FileNotFoundError(f"FlyWire classification not found: {cls_path}")
    cls = pd.read_csv(cls_path, dtype={"root_id": "int64"})

    alpn = cls[(cls["class"] == "ALPN") &
               (cls["sub_class"] == "uniglomerular")].copy()   # uniglomerular only
    alpn["glomerulus"] = alpn["hemibrain_type"].astype(str).map(glom_from_alpn)
    alpn["side"]       = alpn["side"].map(norm_side)
    alpn = alpn.dropna(subset=["glomerulus"])
    alpn = alpn[alpn["glomerulus"].map(is_uniglom)]

    orn = cls[cls["hemibrain_type"].astype(str).str.startswith("ORN_")].copy()
    orn["glomerulus"] = orn["hemibrain_type"].astype(str).map(glom_from_orn)
    orn = orn.dropna(subset=["glomerulus"])
    orn = orn[orn["glomerulus"].map(is_uniglom)]
    orn_glom = dict(zip(orn["root_id"], orn["glomerulus"]))

    if GLOMERULUS_LIMIT:
        keep = sorted(set(alpn["glomerulus"]))[:GLOMERULUS_LIMIT]
        alpn = alpn[alpn["glomerulus"].isin(keep)]
        orn  = orn[orn["glomerulus"].isin(keep)]
        print(f"  [debug] limited to glomeruli: {keep}")

    # root_id → (glomerulus, side) for the unilateral ALPNs
    alpn_info = {r: (g, s) for r, g, s in
                 zip(alpn["root_id"], alpn["glomerulus"], alpn["side"])}
    alpn_ids = alpn["root_id"].tolist()
    orn_ids  = orn["root_id"].tolist()
    print(f"  ALPNs: {len(alpn_ids)} ({alpn['glomerulus'].nunique()} gloms)  |  "
          f"ORNs: {len(orn_ids)} ({orn['glomerulus'].nunique()} gloms)")

    def _query_batch(batch, filter_col):
        """One CAVE batch with retry/backoff — the materialization server
        intermittently drops large COPY queries with a 500."""
        for attempt in range(CAVE_RETRIES):
            try:
                return cave.materialize.query_table(
                    FW_SYN_TABLE,
                    filter_in_dict={filter_col: batch},
                    select_columns=["pre_pt_root_id", "post_pt_root_id",
                                    "pre_pt_position", "post_pt_position"],
                    materialization_version=FW_MAT_VER)
            except Exception as exc:
                if attempt == CAVE_RETRIES - 1:
                    raise
                wait = 2 ** attempt
                print(f"\n    [retry {attempt+1}/{CAVE_RETRIES-1} in {wait}s] "
                      f"{type(exc).__name__}: {str(exc)[:80]}")
                time.sleep(wait)

    def fetch_syn(id_list, filter_col):
        parts = []
        for i in range(0, len(id_list), BATCH_SIZE):
            batch = id_list[i:i + BATCH_SIZE]
            ch = _query_batch(batch, filter_col)
            for col in ("pre_pt_root_id", "post_pt_root_id"):
                ch[col] = ch[col].astype("int64")
            parts.append(ch)
            print(f"    … {min(i+BATCH_SIZE, len(id_list))}/{len(id_list)}",
                  end="\r")
        print()
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

    def _frame(glom_series, coords) -> pd.DataFrame:
        """Assemble a [glomerulus, side, x, y, z] frame; side from x-location."""
        df = pd.DataFrame({
            "glomerulus": np.asarray(glom_series),
            "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
        }).dropna(subset=["glomerulus"])
        df["side"] = np.where(df["x"] < FW_AL_MIDLINE_X, "L", "R")
        return df.reset_index(drop=True)

    frames: dict[str, pd.DataFrame] = {}

    # ── B1: ORN → ALPN presynapses (ORN outputs, and AL bounds for clipping) ──
    #  ORN presynapses are AL-only; their per-(glomerulus, side) bounding box
    #  defines the antennal-lobe region used to clip the PN-input cloud below.
    #  Fetched whenever PN inputs are requested too, since FlyWire synapses carry
    #  no ROI tag (unlike neuprint) and we have no other AL spatial restriction.
    print("  [B1] ORN→ALPN presynaptic sites …")
    raw = fetch_syn(orn_ids, "pre_pt_root_id")
    bbox: dict = {}
    if len(raw):
        raw = raw[raw["post_pt_root_id"].isin(alpn_info)]
        # Glomerulus from the ORN's own identity (avoids cross-glomerular
        # ORN→PN synapses scattering points); side from the synapse location.
        coords = np.vstack(raw["pre_pt_position"].values).astype(np.float64)
        orn_df = _frame(raw["pre_pt_root_id"].map(orn_glom).values, coords)
        for (g, s), grp in orn_df.groupby(["glomerulus", "side"]):
            p = grp[["x", "y", "z"]].values
            lo, hi = p.min(0), p.max(0)
            mar = (hi - lo) * 0.10
            bbox[(g, s)] = (lo - mar, hi + mar)
        if method in ("orn_outputs", "both"):
            orn_df.to_csv(OUTPUT_DIR / "fafb_orn_outputs.csv", index=False)
            frames["orn_outputs"] = orn_df
            print(f"    {len(orn_df):,} ORN presynapses")

    # ── B2: ALPN postsynapses (PN inputs), clipped to each glomerulus's AL box ─
    if method in ("pn_inputs", "both"):
        print("  [B2] ALPN postsynaptic sites …")
        raw = fetch_syn(alpn_ids, "post_pt_root_id")
        if len(raw):
            coords = np.vstack(raw["post_pt_position"].values).astype(np.float64)
            df = _frame(raw["post_pt_root_id"].map(
                lambda r: alpn_info[r][0]).values, coords)
            if bbox:
                keep  = np.zeros(len(df), dtype=bool)
                pts   = df[["x", "y", "z"]].values
                gcol  = df["glomerulus"].values
                scol  = df["side"].values
                for (g, s), (lo, hi) in bbox.items():
                    m = (gcol == g) & (scol == s)
                    if m.any():
                        keep[m] = np.all((pts[m] >= lo) & (pts[m] <= hi), axis=1)
                n0 = len(df); df = df[keep].reset_index(drop=True)
                print(f"    AL-clip kept {len(df):,}/{n0:,} (dropped LH/calyx)")
            else:
                print("    [warn] no ORN bbox — PN inputs left un-clipped")
            df.to_csv(OUTPUT_DIR / "fafb_pn_inputs.csv", index=False)
            frames["pn_inputs"] = df
            print(f"    {len(df):,} synapses")

    return frames


# ══════════════════════════════════════════════════════════════════════════════
# EXPORT + ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def export_hulls(glom: dict, tag: str):
    hull_dir = OUTPUT_DIR / f"{tag}_hull_meshes"
    hull_dir.mkdir(exist_ok=True)
    summary = {}
    for (g, side), d in glom.items():
        key   = f"{g}__AL_{side}"
        verts = d["pts"][d["hull"].vertices]
        np.save(hull_dir / f"{key}_vertices.npy", verts)
        np.save(hull_dir / f"{key}_faces.npy",    d["hull"].simplices)
        np.save(hull_dir / f"{key}_all_pts.npy",  d["raw_pts"])
        summary[key] = dict(
            glomerulus=g, side=side,
            centroid_nm=d["centroid_nm"].tolist(),
            volume_um3=round(d["vol_um3"], 3),
            surface_um2=round(d["area_um2"], 3),
            n_synapses=d["n_syn"],
            n_hull_verts=int(len(verts)),
            n_hull_faces=int(len(d["hull"].simplices)),
        )
    with open(hull_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"    Mesh files → {hull_dir}/")
    return summary


def build_table(glom: dict, method: str) -> pd.DataFrame:
    rows = []
    for (g, side), d in glom.items():
        rows.append(dict(
            glomerulus=g, side=side, method=method,
            centroid_x_nm=round(d["centroid_nm"][0], 1),
            centroid_y_nm=round(d["centroid_nm"][1], 1),
            centroid_z_nm=round(d["centroid_nm"][2], 1),
            volume_um3=round(d["vol_um3"], 4),
            surface_um2=round(d["area_um2"], 4),
            n_synapses=d["n_syn"],
        ))
    return (pd.DataFrame(rows).sort_values(["side", "glomerulus"])
            if rows else pd.DataFrame(
                columns=["glomerulus", "side", "method", "volume_um3"]))


def plot_glomeruli_3d(glom: dict, title: str, filepath: Path,
                      alpha: float = 0.18, max_show: int = 80):
    if not glom:
        return
    items  = list(glom.items())[:max_show]
    colors = cm.tab20(np.linspace(0, 1, max(len(items), 1)))
    fig = plt.figure(figsize=(14, 11))
    ax  = fig.add_subplot(111, projection="3d")
    for color, ((g, side), d) in zip(colors, items):
        faces = [d["pts"][s] for s in d["hull"].simplices]
        ax.add_collection3d(Poly3DCollection(
            faces, alpha=alpha, facecolor=color, edgecolor=(0, 0, 0, 0.05)))
        ax.scatter(*d["centroid"], c=[color], s=28, zorder=5)
        ax.text(*d["centroid"], g, fontsize=5, ha="center")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(filepath, dpi=170, bbox_inches="tight")
    plt.close()
    print(f"    3-D plot → {filepath.name}")


def plot_volume_comparison(pn_tbl, orn_tbl, label, side):
    m = pd.merge(
        pn_tbl[pn_tbl["side"] == side][["glomerulus", "volume_um3"]]
            .rename(columns={"volume_um3": "PN_input"}),
        orn_tbl[orn_tbl["side"] == side][["glomerulus", "volume_um3"]]
            .rename(columns={"volume_um3": "ORN_output"}),
        on="glomerulus", how="inner").sort_values("PN_input", ascending=False)
    if m.empty:
        return
    x = np.arange(len(m))
    fig, ax = plt.subplots(figsize=(max(10, len(m) * 0.45), 5))
    ax.bar(x - 0.2, m["PN_input"],   0.38, label="PN input (postsynaptic)",  color="#4E91C2")
    ax.bar(x + 0.2, m["ORN_output"], 0.38, label="ORN output (presynaptic)", color="#E07B54")
    ax.set_xticks(x); ax.set_xticklabels(m["glomerulus"], rotation=90, fontsize=6.5)
    ax.set_ylabel("Convex-hull volume (µm³)")
    ax.set_title(f"{label}  AL_{side}  — PN inputs vs ORN outputs")
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = OUTPUT_DIR / f"{label.lower()}_AL_{side}_comparison.png"
    plt.savefig(out, dpi=150); plt.close()
    print(f"    Comparison plot → {out.name}")


def process(label: str, syn_frames: dict, coord_unit_nm: float) -> dict:
    print(f"\n  ── {label}: computing glomerulus volumes ──")
    tables: dict[str, pd.DataFrame] = {}
    for method, df in syn_frames.items():
        if df is None or df.empty:
            continue
        print(f"  [{method}]  {len(df):,} synapses  "
              f"({df['glomerulus'].nunique()} glomeruli, sides {sorted(df['side'].unique())})")
        glom = compute_glom_volumes(df, coord_unit_nm=coord_unit_nm)
        print(f"    → {len(glom)} (glomerulus, side) hulls computed")
        tag = f"{label}_{method}"
        tbl = build_table(glom, method)
        tbl.to_csv(OUTPUT_DIR / f"{tag}_volumes.csv", index=False)
        print(f"    CSV → {tag}_volumes.csv")
        export_hulls(glom, tag)
        for side in sorted(df["side"].unique()):
            sub = {k: v for k, v in glom.items() if k[1] == side}
            plot_glomeruli_3d(sub, title=f"{label}  AL_{side}  {method}",
                              filepath=OUTPUT_DIR / f"{tag}_AL_{side}.png")
        tables[method] = tbl

    if "pn_inputs" in tables and "orn_outputs" in tables:
        sides = sorted(set(pd.concat(tables.values())["side"]))
        for side in sides:
            plot_volume_comparison(tables["pn_inputs"], tables["orn_outputs"],
                                   label, side)
    return tables


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    all_results: dict = {}
    if "MCNS" in RUN_DATASETS:
        all_results["MCNS"] = process("MCNS", run_neuprint(SYNAPSE_METHOD),
                                      coord_unit_nm=NP_VOXEL_NM)
    if "FAFB" in RUN_DATASETS:
        all_results["FAFB"] = process("FAFB", run_flywire(SYNAPSE_METHOD),
                                      coord_unit_nm=1.0)

    print(f"\n{'═'*64}\n  Output directory: {OUTPUT_DIR.resolve()}\n{'═'*64}")
    for ds, tbls in all_results.items():
        for method, tbl in tbls.items():
            if len(tbl):
                print(f"  {ds:6s}  {method:14s}  {len(tbl):3d} (glom,side)   "
                      f"vol {tbl['volume_um3'].min():.1f}–{tbl['volume_um3'].max():.1f} µm³")
    return all_results


In [2]:
# Execute end-to-end (edit RUN_DATASETS / SYNAPSE_METHOD / GLOMERULUS_LIMIT above to scope)
results = main()


────────────────────────────────────────────────────────────────
  MCNS  ·  https://neuprint.janelia.org  ·  male-cns:v0.9
────────────────────────────────────────────────────────────────
  Connected  ✓
  ALPNs: 295 (56 gloms)  |  ORNs: 2635 (53 gloms)
  [A1] ALPN postsynaptic sites …


  0%|          | 0/30 [00:00<?, ?it/s]

    874,587 synapses
  [A2] ORN presynaptic sites in AL …


  0%|          | 0/264 [00:00<?, ?it/s]

    414,448 ORN presynapses

  ── MCNS: computing glomerulus volumes ──
  [pn_inputs]  874,587 synapses  (56 glomeruli, sides ['L', 'R'])
    → 112 (glomerulus, side) hulls computed
    CSV → MCNS_pn_inputs_volumes.csv
    Mesh files → glomerulus_output/MCNS_pn_inputs_hull_meshes/
    3-D plot → MCNS_pn_inputs_AL_L.png
    3-D plot → MCNS_pn_inputs_AL_R.png
  [orn_outputs]  414,448 synapses  (53 glomeruli, sides ['L', 'R'])
    → 106 (glomerulus, side) hulls computed
    CSV → MCNS_orn_outputs_volumes.csv
    Mesh files → glomerulus_output/MCNS_orn_outputs_hull_meshes/
    3-D plot → MCNS_orn_outputs_AL_L.png
    3-D plot → MCNS_orn_outputs_AL_R.png
    Comparison plot → mcns_AL_L_comparison.png
    Comparison plot → mcns_AL_R_comparison.png

────────────────────────────────────────────────────────────────
  FAFB  ·  flywire_fafb_public  ·  mat 783
────────────────────────────────────────────────────────────────
  Connected  ✓
  ALPNs: 269 (55 gloms)  |  ORNs: 2276 (51 gloms)
  [B1] OR

...

Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


    … 269/269
    AL-clip kept 607,093/886,081 (dropped LH/calyx)
    607,093 synapses

  ── FAFB: computing glomerulus volumes ──
  [orn_outputs]  243,969 synapses  (51 glomeruli, sides ['L', 'R'])
    → 102 (glomerulus, side) hulls computed
    CSV → FAFB_orn_outputs_volumes.csv
    Mesh files → glomerulus_output/FAFB_orn_outputs_hull_meshes/
    3-D plot → FAFB_orn_outputs_AL_L.png
    3-D plot → FAFB_orn_outputs_AL_R.png
  [pn_inputs]  607,093 synapses  (51 glomeruli, sides ['L', 'R'])
    → 102 (glomerulus, side) hulls computed
    CSV → FAFB_pn_inputs_volumes.csv
    Mesh files → glomerulus_output/FAFB_pn_inputs_hull_meshes/
    3-D plot → FAFB_pn_inputs_AL_L.png
    3-D plot → FAFB_pn_inputs_AL_R.png
    Comparison plot → fafb_AL_L_comparison.png
    Comparison plot → fafb_AL_R_comparison.png

════════════════════════════════════════════════════════════════
  Output directory: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/analysis/glomerulus_output
═════════════

# Bilateral consensus + cross-dataset overlay (ORN outputs)

Collapse the left and right antennal lobes of each dataset into a single
**consensus** AL per glomerulus, then place the MCNS and FAFB consensus ALs
together in **FAFB coordinate space** for direct comparison.

**Pipeline** (ORN outputs only — uses `*_orn_outputs.csv` written above, or run
`main()` first):

1. **Bilateral consensus (per dataset).** Mirror the left AL across the dataset
   midline, fit one 12-DOF affine that maps the mirrored-left glomerulus
   centroids onto the matching right centroids, apply it to all mirrored-left
   points, then pool registered-left + native-right points per glomerulus
   (everything now in the dataset's right-AL frame).
2. **Exclusive consensus volumes.** Voxelize each pooled cloud, fill interiors,
   and resolve voxels claimed by >1 glomerulus to the **nearest centroid** so no
   region of space belongs to two glomeruli. Volume = exclusive voxel count.
3. **Cross-dataset overlay.** Fit one 12-DOF affine mapping MCNS consensus
   centroids → FAFB consensus centroids; keep FAFB at its native right AL and
   mirror the FAFB-frame MCNS across the FAFB midline so it lands where FAFB's
   **left** AL sits. Result: FAFB-derived AL on the right, MCNS-derived AL on
   the left, mirror-posed.

Spec: `docs/superpowers/specs/2026-06-26-bilateral-consensus-overlay-design.md`.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# BILATERAL CONSENSUS + CROSS-DATASET OVERLAY  (ORN outputs)
# ══════════════════════════════════════════════════════════════════════════════
from scipy import ndimage

# ── Consensus config ──────────────────────────────────────────────────────────
CONS_VOXEL_UM   = 2.0      # isotropic voxel size for the exclusive volume grid
CONS_MIN_SYN    = MIN_SYNAPSES   # skip a (glom, side) cloud with fewer points
CONS_RADIAL_Q   = 0.02     # drop farthest q of each cloud (radial) before use
CONS_PLOT_PTS   = 400      # points/glom subsampled into the 3-D overlay scatter


def _orn_csv_for(ds: str) -> Path:
    """Locate the ORN-output CSV for a dataset, searching OUTPUT_DIR then the
    sibling repo-root ``glomerulus_output/`` (where a prior run may have left it)."""
    fn = "mcns_orn_outputs.csv" if ds == "MCNS" else "fafb_orn_outputs.csv"
    for base in (OUTPUT_DIR, Path.cwd() / "glomerulus_output",
                 Path.cwd().parent / "glomerulus_output"):
        if (base / fn).exists():
            return base / fn
    raise FileNotFoundError(f"{fn} not found — run main() with orn_outputs first")


def load_orn_nm(ds: str) -> pd.DataFrame:
    """ORN-output point cloud for ``ds`` with x/y/z in **nm** (MCNS CSV is in
    8-nm voxels; FAFB is already nm)."""
    df = pd.read_csv(_orn_csv_for(ds))
    if ds == "MCNS":
        df[["x", "y", "z"]] = df[["x", "y", "z"]].astype(float) * NP_VOXEL_NM
    return df


def _trim_radial(p: np.ndarray, q: float = CONS_RADIAL_Q) -> np.ndarray:
    """Drop the farthest fraction q of points (Euclidean, from the median)."""
    if len(p) < 10:
        return p
    d = np.linalg.norm(p - np.median(p, axis=0), axis=1)
    return p[d <= np.quantile(d, 1.0 - q)]


def fit_affine(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    """Least-squares 12-DOF affine M (4x3) with  [x, y, z, 1] @ M ≈ dst."""
    S = np.hstack([src, np.ones((len(src), 1))])
    M, *_ = np.linalg.lstsq(S, dst, rcond=None)
    return M


def apply_affine(P: np.ndarray, M: np.ndarray) -> np.ndarray:
    return np.hstack([P, np.ones((len(P), 1))]) @ M


def _rmse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))


def _mirror_x(p: np.ndarray, midline_x: float) -> np.ndarray:
    """Reflect across the AL midline plane x = midline_x."""
    q = p.copy()
    q[:, 0] = 2.0 * midline_x - q[:, 0]
    return q


def bilateral_consensus(df: pd.DataFrame, midline_x: float, ds: str):
    """Mirror L across the midline, affine-register mirrored-L → R glomerulus
    centroids, and pool both sides per glomerulus into the dataset's R-AL frame.

    Returns (pooled, centroids, qc):
      pooled    : {glom -> Nx3 points (nm)} in the right-AL frame
      centroids : {glom -> 3-vector consensus centroid (nm)}
      qc        : dict of registration diagnostics
    """
    clouds = {}
    for (g, s), grp in df.groupby(["glomerulus", "side"]):
        p = grp[["x", "y", "z"]].values.astype(np.float64)
        if len(p) >= CONS_MIN_SYN:
            clouds[(g, s)] = _trim_radial(p)
    cen = {k: v.mean(0) for k, v in clouds.items()}

    gl_R = {g for (g, s) in clouds if s == "R"}
    gl_L = {g for (g, s) in clouds if s == "L"}
    shared = sorted(gl_R & gl_L)

    srcC = np.array([_mirror_x(cen[(g, "L")][None], midline_x)[0] for g in shared])
    dstC = np.array([cen[(g, "R")] for g in shared])
    M = fit_affine(srcC, dstC)

    pooled = {}
    for g in sorted(gl_R | gl_L):
        parts = []
        if (g, "R") in clouds:
            parts.append(clouds[(g, "R")])
        if (g, "L") in clouds:
            parts.append(apply_affine(_mirror_x(clouds[(g, "L")], midline_x), M))
        pooled[g] = np.vstack(parts)
    centroids = {g: p.mean(0) for g, p in pooled.items()}
    qc = dict(dataset=ds, n_shared=len(shared), n_gloms=len(pooled),
              midline_x_nm=float(midline_x),
              rmse_pre_nm=round(_rmse(srcC, dstC), 1),
              rmse_post_nm=round(_rmse(apply_affine(srcC, M), dstC), 1))
    return pooled, centroids, qc


def exclusive_volumes(pooled: dict, voxel_um: float = CONS_VOXEL_UM):
    """Voxelize pooled clouds on a shared grid, fill interiors, and resolve
    voxels claimed by >1 glomerulus to the nearest glomerulus centroid so the
    partition is exclusive.  Returns (vol_um3 dict, grid-info dict)."""
    voxel_nm = voxel_um * 1000.0
    gloms = sorted(pooled)
    allp  = np.vstack([pooled[g] for g in gloms])
    lo    = allp.min(0) - voxel_nm
    dims  = np.ceil((allp.max(0) + voxel_nm - lo) / voxel_nm).astype(int)
    cen   = {g: pooled[g].mean(0) for g in gloms}

    masks = np.zeros((len(gloms), *dims), dtype=bool)
    for gi, g in enumerate(gloms):
        idx = np.floor((pooled[g] - lo) / voxel_nm).astype(int)
        m = np.zeros(dims, bool)
        m[idx[:, 0], idx[:, 1], idx[:, 2]] = True
        m = ndimage.binary_closing(m, iterations=1)
        m = ndimage.binary_fill_holes(m)
        masks[gi] = m

    count = masks.sum(0)
    owner = np.full(dims, -1, np.int32)
    single = count == 1
    owner[single] = masks[:, single].argmax(0)

    cvox = np.argwhere(count > 1)               # contested voxels
    if len(cvox):
        world = lo + (cvox + 0.5) * voxel_nm
        cmask = masks[:, cvox[:, 0], cvox[:, 1], cvox[:, 2]]      # (G, ncon)
        cenA  = np.array([cen[g] for g in gloms])                # (G, 3)
        d = np.linalg.norm(world[None, :, :] - cenA[:, None, :], axis=2)
        d[~cmask] = np.inf                       # only gloms that claim the voxel
        owner[cvox[:, 0], cvox[:, 1], cvox[:, 2]] = d.argmin(0)

    vcell = voxel_um ** 3
    vol = {g: int((owner == gi).sum()) * vcell for gi, g in enumerate(gloms)}
    return vol, dict(lo=lo, dims=dims, voxel_nm=voxel_nm, owner=owner,
                     gloms=gloms, cen=cen, n_contested=int((count > 1).sum()))


def plot_consensus_overlay(mcns_pts: dict, fafb_pts: dict, filepath: Path,
                           n_pts: int = CONS_PLOT_PTS):
    """3-D scatter of the FAFB-space overlay: FAFB consensus (●) + MCNS
    consensus mirrored to the opposite AL (▲), colour-keyed by glomerulus."""
    allg = sorted(set(mcns_pts) | set(fafb_pts))
    cmap = {g: cm.tab20(i / max(len(allg), 1)) for i, g in enumerate(allg)}
    rng  = np.random.default_rng(0)
    fig = plt.figure(figsize=(15, 8))
    ax  = fig.add_subplot(111, projection="3d")
    for d, marker in ((fafb_pts, "o"), (mcns_pts, "^")):
        for g, p in d.items():
            s = p[rng.choice(len(p), min(len(p), n_pts), replace=False)]
            ax.scatter(s[:, 0], s[:, 1], s[:, 2], s=2, marker=marker,
                       color=cmap[g], alpha=0.3)
    ax.scatter([], [], c="k", marker="o", label="FAFB consensus (right AL)")
    ax.scatter([], [], c="k", marker="^", label="MCNS consensus (left AL)")
    ax.legend(loc="upper left", fontsize=9)
    ax.set_xlabel("X (nm)"); ax.set_ylabel("Y (nm)"); ax.set_zlabel("Z (nm)")
    ax.set_title("FAFB-space overlay — FAFB (right) vs MCNS (left), mirror-posed",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.savefig(filepath, dpi=150); plt.close()
    print(f"    3-D overlay → {filepath.name}")


def plot_consensus_volume_scatter(vol_m: dict, vol_f: dict, filepath: Path):
    """Per-glomerulus exclusive consensus volume, MCNS vs FAFB."""
    sh = sorted(set(vol_m) & set(vol_f))
    if not sh:
        return
    xm = np.array([vol_m[g] for g in sh]); ym = np.array([vol_f[g] for g in sh])
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    ax.scatter(xm, ym, s=28, color="#4E91C2")
    mx = max(xm.max(), ym.max()) * 1.05
    ax.plot([0, mx], [0, mx], "k--", lw=0.8, label="y = x")
    for g in sh:
        ax.annotate(g, (vol_m[g], vol_f[g]), fontsize=5)
    ax.set_xlabel("MCNS consensus volume (µm³)")
    ax.set_ylabel("FAFB consensus volume (µm³)")
    ax.set_title("Exclusive consensus volume per glomerulus")
    ax.set_aspect("equal"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(filepath, dpi=150); plt.close()
    print(f"    Volume scatter → {filepath.name}")


def run_consensus():
    """End-to-end: per-dataset bilateral consensus → exclusive volumes →
    MCNS↔FAFB overlay in FAFB space.  Writes CSV / JSON / PNG to OUTPUT_DIR."""
    cons = {}
    qc_all = []
    for ds in ("MCNS", "FAFB"):
        df = load_orn_nm(ds)
        midline = (FW_AL_MIDLINE_X if ds == "FAFB"
                   else float(np.median(df["x"].values)))   # MCNS: data midline
        pooled, cen, qc = bilateral_consensus(df, midline, ds)
        vol, grid = exclusive_volumes(pooled)
        cons[ds] = dict(pooled=pooled, cen=cen, vol=vol, grid=grid, midline=midline)
        qc_all.append(qc)
        print(f"  [{ds}] shared={qc['n_shared']}  L→R RMSE "
              f"{qc['rmse_pre_nm']:.0f}→{qc['rmse_post_nm']:.0f} nm  "
              f"gloms={qc['n_gloms']}  contested_vox={grid['n_contested']}  "
              f"vol {min(vol.values()):.0f}–{max(vol.values()):.0f} µm³")

    # ── cross-dataset: MCNS consensus centroids → FAFB consensus centroids ────
    mc, fc = cons["MCNS"]["cen"], cons["FAFB"]["cen"]
    shared = sorted(set(mc) & set(fc))
    srcC = np.array([mc[g] for g in shared]); dstC = np.array([fc[g] for g in shared])
    Mx = fit_affine(srcC, dstC)
    qc_x = dict(step="MCNS→FAFB", n_shared=len(shared),
                rmse_pre_nm=round(_rmse(srcC, dstC), 1),
                rmse_post_nm=round(_rmse(apply_affine(srcC, Mx), dstC), 1))
    print(f"  [X] MCNS→FAFB shared={len(shared)}  RMSE "
          f"{qc_x['rmse_pre_nm']:.0f}→{qc_x['rmse_post_nm']:.0f} nm")

    # FAFB stays at its native (right-AL) frame; MCNS → FAFB frame, then mirror
    # across the FAFB midline so it lands where FAFB's *left* AL sits.
    fmid = cons["FAFB"]["midline"]
    mcns_fafb = {g: _mirror_x(apply_affine(p, Mx), fmid)
                 for g, p in cons["MCNS"]["pooled"].items()}
    fafb_R = cons["FAFB"]["pooled"]

    # ── outputs ───────────────────────────────────────────────────────────────
    rows = []
    for ds in ("MCNS", "FAFB"):
        for g, v in cons[ds]["vol"].items():
            c = cons[ds]["cen"][g]
            rows.append(dict(dataset=ds, glomerulus=g, volume_um3=round(v, 3),
                             centroid_x_nm=round(c[0], 1), centroid_y_nm=round(c[1], 1),
                             centroid_z_nm=round(c[2], 1)))
    vol_tbl = pd.DataFrame(rows).sort_values(["dataset", "glomerulus"])
    vol_tbl.to_csv(OUTPUT_DIR / "consensus_volumes.csv", index=False)
    print(f"    CSV → consensus_volumes.csv  ({len(vol_tbl)} rows)")

    with open(OUTPUT_DIR / "consensus_registration_qc.json", "w") as f:
        json.dump({"within_dataset": qc_all, "cross_dataset": qc_x,
                   "voxel_um": CONS_VOXEL_UM}, f, indent=2)
    print("    QC  → consensus_registration_qc.json")

    plot_consensus_overlay(mcns_fafb, fafb_R, OUTPUT_DIR / "consensus_overlay_3d.png")
    plot_consensus_volume_scatter(cons["MCNS"]["vol"], cons["FAFB"]["vol"],
                                  OUTPUT_DIR / "consensus_volume_scatter.png")
    return dict(cons=cons, overlay=dict(mcns=mcns_fafb, fafb=fafb_R),
                Mx=Mx, vol_tbl=vol_tbl, qc=dict(within=qc_all, cross=qc_x))

In [4]:
# Run the bilateral consensus + cross-dataset overlay (reads *_orn_outputs.csv).
consensus = run_consensus()

  [MCNS] shared=53  L→R RMSE 57430→4300 nm  gloms=53  contested_vox=3847  vol 1232–17440 µm³
  [FAFB] shared=51  L→R RMSE 22874→3643 nm  gloms=51  contested_vox=1912  vol 1000–12056 µm³
  [X] MCNS→FAFB shared=49  RMSE 271295→4051 nm
    CSV → consensus_volumes.csv  (104 rows)
    QC  → consensus_registration_qc.json
    3-D overlay → consensus_overlay_3d.png
    Volume scatter → consensus_volume_scatter.png


In [ ]:
# ── Save consensus 3-D hulls in the common (FAFB) space ───────────────────────
# One convex-hull mesh per (dataset, glomerulus), placed in the shared FAFB
# space (FAFB → right AL, MCNS → mirrored to left AL), tagged with glomerular
# identity so they can be reloaded later and coloured by any property (e.g. for
# rotating videos).  Vertices are nm; faces index into the SAVED vertices
# (self-contained — load verts[faces] straight into a Poly3DCollection).
#
# Also saves the exclusive voxel owner-grid per dataset (compact .npz) so
# strictly non-overlapping per-glomerulus meshes can be regenerated on demand.

def export_consensus_hulls(consensus: dict, out_dir: Path = None) -> pd.DataFrame:
    out_dir = out_dir or (OUTPUT_DIR / "consensus_hull_meshes")
    out_dir.mkdir(parents=True, exist_ok=True)

    # common-space point clouds already assembled by run_consensus()
    common = {("FAFB", g): p for g, p in consensus["overlay"]["fafb"].items()}
    common.update({("MCNS", g): p for g, p in consensus["overlay"]["mcns"].items()})
    al_side    = {"FAFB": "R", "MCNS": "L"}     # AL each lands on in FAFB space
    vol_native = {ds: consensus["cons"][ds]["vol"] for ds in ("MCNS", "FAFB")}

    manifest, packed = [], {}
    for (ds, g), pts in sorted(common.items()):
        p = _trim_radial(pts.astype(np.float64))
        if len(p) < 4:
            continue
        try:
            hull = ConvexHull(p)
        except Exception as exc:
            print(f"    hull failed {ds}/{g}: {exc}");  continue
        # compact self-contained mesh: re-index simplices into the kept vertices
        used  = np.unique(hull.simplices)
        remap = np.full(used.max() + 1, -1, np.int64); remap[used] = np.arange(len(used))
        verts = p[used]
        faces = remap[hull.simplices]

        key = f"{ds}__{g}"
        np.save(out_dir / f"{key}_vertices.npy", verts)
        np.save(out_dir / f"{key}_faces.npy", faces)
        packed[f"{key}__verts"] = verts.astype(np.float32)
        packed[f"{key}__faces"] = faces.astype(np.int32)
        c = verts.mean(0)
        manifest.append(dict(
            dataset=ds, glomerulus=g, common_space="FAFB", al_side=al_side[ds],
            centroid_x_nm=round(float(c[0]), 1), centroid_y_nm=round(float(c[1]), 1),
            centroid_z_nm=round(float(c[2]), 1),
            n_verts=int(len(verts)), n_faces=int(len(faces)),
            hull_vol_um3_common=round(hull.volume / 1e9, 3),    # in FAFB-scaled space
            excl_vol_um3_native=round(float(vol_native[ds].get(g, np.nan)), 3),
        ))

    mani = pd.DataFrame(manifest).sort_values(["dataset", "glomerulus"])
    mani.to_json(out_dir / "manifest.json", orient="records", indent=2)
    np.savez_compressed(out_dir / "consensus_hull_meshes.npz", **packed)

    # exclusive voxel owner-grid per dataset (common space) for non-overlapping meshes
    for ds in ("FAFB", "MCNS"):
        cdict = {g: consensus["overlay"]["fafb" if ds == "FAFB" else "mcns"][g]
                 for g in consensus["cons"][ds]["pooled"]}
        _, grid = exclusive_volumes(cdict)
        np.savez_compressed(
            out_dir / f"consensus_voxel_grid_{ds}.npz",
            owner=grid["owner"].astype(np.int16), origin_nm=grid["lo"],
            voxel_nm=np.float64(grid["voxel_nm"]),
            gloms=np.array(grid["gloms"], dtype=object))

    print(f"    {len(mani)} consensus hull meshes → {out_dir}/")
    print(f"    Packed → {out_dir.name}/consensus_hull_meshes.npz  +  manifest.json")
    print(f"    Exclusive voxel owner-grids → consensus_voxel_grid_{{FAFB,MCNS}}.npz")
    return mani


consensus_hulls = export_consensus_hulls(consensus)
consensus_hulls.groupby("dataset").size()